# 04 · Comparación de modelos Keras para ASL

Este notebook prepara una comparación reproducible entre la CNN propia, ResNet-50V2 y EfficientNetB0. Los tres modelos usan las mismas particiones, imágenes de 200 × 200, tamaño de lote y criterio de Early Stopping. ResNet-50V2 y EfficientNetB0 empiezan con pesos preentrenados de ImageNet y una base congelada; se entrena primero su clasificador superior.

**Ejecútalo en el clúster con TensorFlow/GPU cuando tengas acceso.** Los pesos ImageNet podrían descargarse la primera vez, así que el entorno necesita acceso a internet o una caché previa. El dataset no está en Git y debe estar disponible en `data/raw/` en el clúster.

La configuración está en `configs/baseline.json` y `configs/comparison_keras.json`. El ajuste y la evaluación final están desactivados por defecto para evitar iniciar trabajos largos al ejecutar todas las celdas. No cambies hiperparámetros basándote en el conjunto de prueba.

In [ ]:
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

_here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [_here, *_here.parents] if (p / "src").is_dir() and (p / "configs").is_dir()), _here)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import create_stratified_splits, build_tf_dataset
from src.model import build_keras_model, compile_classifier
from src.train import set_random_seeds, train_model
from src.evaluate import evaluate_classifier

with (PROJECT_ROOT / "configs" / "baseline.json").open(encoding="utf-8") as file:
    baseline_config = json.load(file)
with (PROJECT_ROOT / "configs" / "comparison_keras.json").open(encoding="utf-8") as file:
    comparison_config = json.load(file)

CLASS_ROOT = PROJECT_ROOT / "data" / "raw" / "asl_alphabet_train" / "asl_alphabet_train"
set_random_seeds(baseline_config["seed"])
print("Proyecto:", PROJECT_ROOT)
print("GPU visibles para TensorFlow:", tf.config.list_physical_devices("GPU"))
print("Modelos preparados:", comparison_config["models_to_run"])
print("Raíz de clases:", CLASS_ROOT)
if not CLASS_ROOT.is_dir():
    raise FileNotFoundError(f"No se encontró {CLASS_ROOT}; coloca el dataset bajo data/raw en este entorno.")

## 1. Crear las particiones

Se aplica la misma semilla y proporción usada en el notebook 02. El cálculo de SHA-256 revisa todas las imágenes antes de separar para evitar copias exactas entre particiones; esta preparación puede tardar.

In [ ]:
splits = create_stratified_splits(
    CLASS_ROOT,
    validation_fraction=baseline_config["validation_fraction"],
    test_fraction=baseline_config["test_fraction"],
    seed=baseline_config["seed"],
)
image_size = tuple(baseline_config["image_size"])
batch_size = baseline_config["batch_size"]
train_ds = build_tf_dataset(splits.train, image_size=image_size, batch_size=batch_size, shuffle=True, seed=baseline_config["seed"])
validation_ds = build_tf_dataset(splits.validation, image_size=image_size, batch_size=batch_size, shuffle=False)
test_ds = build_tf_dataset(splits.test, image_size=image_size, batch_size=batch_size, shuffle=False)
print(f"train={len(splits.train):,} | validation={len(splits.validation):,} | test reservado={len(splits.test):,}")
print("Clases:", len(splits.class_names), "| duplicados internos removidos:", splits.duplicates_removed)

## 2. Entrenar los modelos

Cambia `RUN_TRAINING` a `True` y ejecuta esta celda de forma intencional en el clúster. Cada modelo se guarda después de cada mejora en `val_loss`; su historial CSV y un resumen de tiempo/tamaño se guardan bajo `results/metrics/`. Si el trabajo se interrumpe, el modelo que ya terminó conserva su checkpoint.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    training_summaries = {}
    for model_name in comparison_config["models_to_run"]:
        print(f"\n===== Entrenando {model_name} =====")
        set_random_seeds(baseline_config["seed"])
        model = build_keras_model(
            model_name,
            input_shape=(*image_size, 3),
            num_classes=len(splits.class_names),
            dropout_rate=baseline_config["dropout_rate"],
            weights=comparison_config["pretrained_weights"],
            train_base=comparison_config["train_base"],
        )
        compile_classifier(model, learning_rate=baseline_config["learning_rate"])
        total_parameters = model.count_params()
        trainable_parameters = sum(int(np.prod(weight.shape)) for weight in model.trainable_weights)
        print(f"Parámetros totales: {total_parameters:,} | entrenables: {trainable_parameters:,}")

        started_at = time.perf_counter()
        history = train_model(
            model,
            train_ds,
            validation_ds,
            epochs=baseline_config["epochs"],
            patience=baseline_config["early_stopping_patience"],
            checkpoint_path=PROJECT_ROOT / comparison_config["checkpoint_paths"][model_name],
            history_path=PROJECT_ROOT / comparison_config["history_paths"][model_name],
        )
        elapsed_seconds = time.perf_counter() - started_at
        history_frame = pd.DataFrame(history.history)
        best_row_index = int(history_frame["val_loss"].idxmin())
        checkpoint = PROJECT_ROOT / comparison_config["checkpoint_paths"][model_name]
        summary = {
            "model": model_name,
            "parameters": total_parameters,
            "trainable_parameters": trainable_parameters,
            "epochs_completed": len(history_frame),
            "best_epoch": best_row_index + 1,
            "best_val_loss": float(history_frame.loc[best_row_index, "val_loss"]),
            "best_val_accuracy": float(history_frame.loc[best_row_index, "val_accuracy"]),
            "training_seconds": elapsed_seconds,
            "checkpoint_size_mb": checkpoint.stat().st_size / (1024 ** 2),
            "checkpoint_path": comparison_config["checkpoint_paths"][model_name],
            "history_path": comparison_config["history_paths"][model_name],
        }
        summary_path = PROJECT_ROOT / comparison_config["training_summary_paths"][model_name]
        summary_path.parent.mkdir(parents=True, exist_ok=True)
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        training_summaries[model_name] = summary
        print(json.dumps(summary, indent=2))
        del model, history
else:
    print("Entrenamiento no iniciado. Cambia RUN_TRAINING a True para ejecutar los modelos configurados.")

## 3. Evaluación final y tabla comparativa

Ejecuta esta sección **después** de completar y fijar las configuraciones de todos los modelos. El conjunto de prueba reservado se utiliza aquí una sola vez para cada modelo y no participa en entrenamiento ni selección del checkpoint. Se comparan accuracy y precision/recall/F1 macro; además, se guardan métricas por clase y matrices de confusión.

In [ ]:
RUN_TEST_EVALUATION = False

if RUN_TEST_EVALUATION:
    model_paths = {
        name: PROJECT_ROOT / path
        for name, path in comparison_config["checkpoint_paths"].items()
    }
    rows = []
    for model_name, checkpoint in model_paths.items():
        if not checkpoint.exists():
            print(f"Se omite {model_name}: todavía no existe {checkpoint.name}.")
            continue
        print(f"Evaluando {model_name} en el test reservado...")
        loaded_model = tf.keras.models.load_model(checkpoint, compile=False)
        test_metrics, per_class_frame, confusion = evaluate_classifier(
            loaded_model, test_ds, splits.class_names
        )
        summary_path = PROJECT_ROOT / comparison_config["training_summary_paths"][model_name]
        train_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
        rows.append({
            "model": model_name,
            **{key: value for key, value in train_summary.items() if key not in {"model", "checkpoint_path", "history_path"}},
            "test_accuracy": test_metrics["accuracy"],
            "test_precision_macro": test_metrics["precision_macro"],
            "test_recall_macro": test_metrics["recall_macro"],
            "test_f1_macro": test_metrics["f1_macro"],
            "test_samples": test_metrics["samples"],
        })
        metrics_dir = PROJECT_ROOT / "results" / "metrics"
        metrics_dir.mkdir(parents=True, exist_ok=True)
        per_class_frame.to_csv(metrics_dir / f"{model_name}_per_class_test.csv")
        pd.DataFrame(confusion, index=splits.class_names, columns=splits.class_names).to_csv(
            metrics_dir / f"{model_name}_confusion_test.csv", index_label="real/predicha"
        )
        del loaded_model

    if rows:
        comparison_table = pd.DataFrame(rows).sort_values("test_f1_macro", ascending=False)
        display(comparison_table)
        table_path = PROJECT_ROOT / comparison_config["comparison_table_path"]
        table_path.parent.mkdir(parents=True, exist_ok=True)
        comparison_table.to_csv(table_path, index=False)
        print("Tabla guardada en:", table_path)
    else:
        print("No hay checkpoints para evaluar todavía. Ejecuta primero el entrenamiento.")
else:
    print("Evaluación no iniciada. Cámbiala a True cuando todos los modelos estén listos para la evaluación final.")

## Archivos generados

- Mejor checkpoint por arquitectura: `models/*_best.keras` (ignorado por Git).
- Historial por época: `results/metrics/*_history.csv`.
- Tiempo, tamaños y mejor época: `results/metrics/*_training_summary.json`.
- Tabla comparativa de prueba: `results/metrics/keras_model_comparison.csv`.
- Métricas por clase y matrices: `results/metrics/*_per_class_test.csv` y `results/metrics/*_confusion_test.csv`.

Los resultados dependen del hardware, versiones y acceso a pesos preentrenados. Conserva la configuración, el log y el entorno utilizado al ejecutar esto en el clúster.